# SmartLeafNet experiments

Exploratory model comparisons for rice leaf disease classification. Outputs and machine-specific paths were removed for the public repository. Dataset paths are relative to `data/train` and `data/valid`.

For the cleaned, leakage-aware implementation, use [`../src/train.py`](../src/train.py).


In [ ]:
import os
import numpy as np
import tensorflow as tf
import gc
import optuna
from tensorflow.keras.applications import EfficientNetB0, ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Paths
train_dir = r"data/train"
valid_dir = r"data/valid"

# Config
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 8
EPOCHS = 50

# Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = len(train_generator.class_indices)
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
train_labels = train_generator.classes

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = dict(enumerate(class_weights))

# Callbacks
early_stopping = EarlyStopping(monitor="val_accuracy", patience=20, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)

# Model Training

def build_model(base_model_class, name):
    print(f"\n🔧 Training {name} for {EPOCHS} epochs...")
    base_model = base_model_class(include_top=False, weights='imagenet', input_shape=input_shape)
    base_model.trainable = False

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = Model(inputs=base_model.input, outputs=out, name=name)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    model.fit(
        train_generator,
        validation_data=valid_generator,
        epochs=EPOCHS,
        callbacks=[early_stopping, reduce_lr],
        class_weight=class_weights,
        verbose=1
    )

    return model

# Feature Extraction

def extract_features(model):
    feature_model = Model(inputs=model.input, outputs=model.layers[-3].output)
    train_features = feature_model.predict(train_generator, verbose=1)
    valid_features = feature_model.predict(valid_generator, verbose=1)
    return train_features, valid_features

# Train and extract features from both models
all_train_feats, all_valid_feats = [], []
for base_model_class in [EfficientNetB0, ResNet50]:
    model = build_model(base_model_class, base_model_class.__name__)
    train_feat, valid_feat = extract_features(model)
    all_train_feats.append(train_feat)
    all_valid_feats.append(valid_feat)
    del model
    tf.keras.backend.clear_session()
    gc.collect()

# Combine features
train_features = np.concatenate(all_train_feats, axis=1)
valid_features = np.concatenate(all_valid_feats, axis=1)

# Preprocessing
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
valid_features = scaler.transform(valid_features)

pca = PCA(n_components=100)
train_features = pca.fit_transform(train_features)
valid_features = pca.transform(valid_features)

sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(train_features, train_labels)
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Optuna Objective

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'use_label_encoder': False
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='mlogloss', early_stopping_rounds=10, verbose=False)
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='macro')

# Optimize with Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# Final Model
best_params = study.best_params
best_params.update({'use_label_encoder': False, 'random_state': 42})
final_model = XGBClassifier(**best_params)
final_model.fit(X_resampled, y_resampled, eval_metric='mlogloss', verbose=False)

# Evaluate
train_preds = final_model.predict(train_features)
valid_preds = final_model.predict(valid_features)

print("\n🔍 Classification Report on Training Data:")
print(classification_report(train_labels, train_preds, target_names=list(train_generator.class_indices.keys())))
print(f"Training Accuracy: {accuracy_score(train_labels, train_preds) * 100:.2f}%")

valid_labels = valid_generator.classes
print("\n📊 Classification Report on Validation Data:")
print(classification_report(valid_labels, valid_preds, target_names=list(valid_generator.class_indices.keys())))
print(f"Validation Accuracy: {accuracy_score(valid_labels, valid_preds) * 100:.2f}%")

## Additional model trials

The cells below preserve the team’s exploratory comparisons of alternative feature extractors and classifiers. They are retained as a research log, not as the canonical reproducible pipeline.


In [ ]:
import os
import numpy as np
import tensorflow as tf
import gc
import optuna
from tensorflow.keras.applications import InceptionV3, DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Paths
train_dir = r"data/train"
valid_dir = r"data/valid"

# Config
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 8
EPOCHS = 1

# Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode="nearest"
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = len(train_generator.class_indices)
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
train_labels = train_generator.classes

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = dict(enumerate(class_weights))

# Callbacks
early_stopping = EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)

# Build model
def build_and_train_model(base_model_class, model_name):
    base_model = base_model_class(weights="imagenet", include_top=False, input_shape=input_shape)
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation="softmax", dtype='float32')(x)

    model = Model(inputs=base_model.input, outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])

    model.fit(
        train_generator,
        validation_data=valid_generator,
        epochs=EPOCHS,
        callbacks=[early_stopping, reduce_lr],
        class_weight=class_weights,
        verbose=1
    )

    return model

def extract_features(model):
    feature_extractor = Model(inputs=model.input, outputs=model.layers[-3].output)
    train_features = feature_extractor.predict(train_generator, verbose=1)
    valid_features = feature_extractor.predict(valid_generator, verbose=1)
    return train_features, valid_features

# Run training and feature extraction
all_train_feats, all_valid_feats = [], []
for base_model_class in [InceptionV3, DenseNet121]:
    model = build_and_train_model(base_model_class, base_model_class.__name__)
    train_feats, valid_feats = extract_features(model)
    all_train_feats.append(train_feats)
    all_valid_feats.append(valid_feats)
    del model
    tf.keras.backend.clear_session()
    gc.collect()

# Concatenate features
train_features = np.concatenate(all_train_feats, axis=1)
valid_features = np.concatenate(all_valid_feats, axis=1)

# Standardize + Dimensionality Reduction
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
valid_features = scaler.transform(valid_features)

pca = PCA(n_components=100)
train_features = pca.fit_transform(train_features)
valid_features = pca.transform(valid_features)

# Resample with SMOTE
sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(train_features, train_labels)
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Optuna objective
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'use_label_encoder': False
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='mlogloss', early_stopping_rounds=10, verbose=False)
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='macro')

# Run Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# Train final model
best_params = study.best_params
best_params.update({'use_label_encoder': False, 'random_state': 42})
final_model = XGBClassifier(**best_params)
final_model.fit(X_resampled, y_resampled, eval_metric='mlogloss', verbose=False)

# Evaluate
train_preds = final_model.predict(train_features)
valid_preds = final_model.predict(valid_features)

print("\n🔍 Classification Report on Training Data:")
print(classification_report(train_labels, train_preds, target_names=list(train_generator.class_indices.keys())))
print(f"Training Accuracy: {accuracy_score(train_labels, train_preds) * 100:.2f}%")

valid_labels = valid_generator.classes
print("\n📊 Classification Report on Validation Data:")
print(classification_report(valid_labels, valid_preds, target_names=list(valid_generator.class_indices.keys())))
print(f"Validation Accuracy: {accuracy_score(valid_labels, valid_preds) * 100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Predictions already obtained
train_preds = final_model.predict(train_features)
valid_preds = final_model.predict(valid_features)

# Accuracy scores
train_acc = accuracy_score(train_labels, train_preds)
valid_acc = accuracy_score(valid_labels, valid_preds)

# Plotting
plt.figure(figsize=(6, 4))
plt.bar(['Training Accuracy', 'Validation Accuracy'], [train_acc * 100, valid_acc * 100])
plt.ylim(0, 100)
plt.ylabel('Accuracy (%)')
plt.title('Training vs Validation Accuracy')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import tensorflow as tf
import gc
import optuna
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

# Paths
train_dir = r"data/train"
valid_dir = r"data/valid"

# Config
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 8
EPOCHS = 50

# Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    zoom_range=0.3,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = len(train_generator.class_indices)
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
train_labels = train_generator.classes

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = dict(enumerate(class_weights))

# Callbacks
early_stopping_cb = EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)

# Build EfficientNetB3 Model
def build_single_model():
    base_model = EfficientNetB3(include_top=False, weights='imagenet', input_shape=input_shape)
    base_model.trainable = False

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = Dropout(0.6)(x)
    out = Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = Model(inputs=base_model.input, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    model.fit(
        train_generator,
        validation_data=valid_generator,
        epochs=EPOCHS,
        callbacks=[early_stopping_cb, reduce_lr],
        class_weight=class_weights,
        verbose=1
    )

    return model

# Train Model & Extract Features
model = build_single_model()
feature_model = Model(inputs=model.input, outputs=model.layers[-3].output)
train_features = feature_model.predict(train_generator, verbose=1)
valid_features = feature_model.predict(valid_generator, verbose=1)

# Preprocessing
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
valid_features = scaler.transform(valid_features)

pca = PCA(n_components=100)
train_features = pca.fit_transform(train_features)
valid_features = pca.transform(valid_features)

# SMOTE and Split
X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(train_features, train_labels)
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

# Optuna Optimization
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }

    model = LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(stopping_rounds=10), log_evaluation(0)]
    )

    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='macro')

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# Train Final Model
best_params = study.best_params
best_params.update({'random_state': 42})
final_model = LGBMClassifier(**best_params)
final_model.fit(X_resampled, y_resampled)

# Evaluate
train_preds = final_model.predict(train_features)
valid_preds = final_model.predict(valid_features)

print("\n🔍 Training Report:")
print(classification_report(train_labels, train_preds, target_names=list(train_generator.class_indices.keys())))
print(f"Training Accuracy: {accuracy_score(train_labels, train_preds) * 100:.2f}%")

print("\n📊 Validation Report:")
print(classification_report(valid_generator.classes, valid_preds, target_names=list(valid_generator.class_indices.keys())))
print(f"Validation Accuracy: {accuracy_score(valid_generator.classes, valid_preds) * 100:.2f}%")

In [ ]:
import os
import numpy as np
import tensorflow as tf
import gc
import joblib
import optuna
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier as XGBSklearn
from lightgbm import LGBMClassifier as LGBMSklearn, early_stopping as lgb_early_stopping
from sklearn.svm import SVC
from sklearn.base import clone  # ✅ Important fix
from tensorflow.keras.applications import EfficientNetB3, DenseNet121, InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

# Enable GPU memory growth
physical_devices = tf.config.list_physical_devices('GPU')
for gpu in physical_devices:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

# Paths
train_dir = r"data/train"
valid_dir = r"data/valid"
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 8
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)

# Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
valid_generator = valid_datagen.flow_from_directory(
    valid_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

num_classes = len(train_generator.class_indices)
train_labels = train_generator.classes

# Class Weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = dict(enumerate(class_weights))

# Feature Extraction
def extract_and_save_features(base_model_class, model_name, generator, save_dir="features"):
    print(f"\n🚀 Extracting features using {model_name}...")
    tf.keras.backend.clear_session()
    gc.collect()
    base_model = base_model_class(include_top=False, weights='imagenet', input_shape=input_shape)
    base_model.trainable = False
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.5)(x)
    model = Model(inputs=base_model.input, outputs=x)
    features = model.predict(generator, verbose=1)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, f"{model_name}_features.npy"), features)
    tf.keras.backend.clear_session()
    del base_model, model
    gc.collect()

# Extract Features (Training + Validation)
extract_and_save_features(EfficientNetB3, "effnet_train", train_generator)
extract_and_save_features(DenseNet121, "densenet_train", train_generator)
extract_and_save_features(InceptionV3, "inception_train", train_generator)
extract_and_save_features(EfficientNetB3, "effnet_valid", valid_generator)
extract_and_save_features(DenseNet121, "densenet_valid", valid_generator)
extract_and_save_features(InceptionV3, "inception_valid", valid_generator)

# Load Features
train_features = np.concatenate([
    np.load("features/effnet_train_features.npy"),
    np.load("features/densenet_train_features.npy"),
    np.load("features/inception_train_features.npy")
], axis=1)

valid_features = np.concatenate([
    np.load("features/effnet_valid_features.npy"),
    np.load("features/densenet_valid_features.npy"),
    np.load("features/inception_valid_features.npy")
], axis=1)

# Scale & Reduce Dimensionality
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
valid_features = scaler.transform(valid_features)

pca = PCA(n_components=150)
train_features = pca.fit_transform(train_features)
valid_features = pca.transform(valid_features)

# Labels
y_train = train_generator.classes
y_valid = valid_generator.classes
X_train, X_val, y_train_split, y_val_split = train_test_split(train_features, y_train, test_size=0.2, stratify=y_train, random_state=42)

# Optuna Tuning Functions
def objective_lgb(trial):
    print(f"\n🔍 Tuning LightGBM - Trial {trial.number + 1}")
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }
    model = LGBMSklearn(**params)
    model.fit(X_train, y_train_split, eval_set=[(X_val, y_val_split)],
              callbacks=[lgb_early_stopping(10)])
    return model.score(X_val, y_val_split)

def objective_xgb(trial):
    print(f"\n🔍 Tuning XGBoost - Trial {trial.number + 1}")
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'use_label_encoder': False,
        'eval_metric': 'mlogloss'
    }
    model = XGBSklearn(**params)
    model.fit(
        X_train,
        y_train_split,
        eval_set=[(X_val, y_val_split)],
        early_stopping_rounds=10
    )
    return model.score(X_val, y_val_split)

def objective_svc(trial):
    print(f"\n🔍 Tuning SVC - Trial {trial.number + 1}")
    params = {
        'C': trial.suggest_float('C', 0.1, 100.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-5, 1.0, log=True),
        'kernel': trial.suggest_categorical('kernel', ['rbf', 'poly']),
        'probability': True,
        'random_state': 42
    }
    model = SVC(**params)
    model.fit(X_train, y_train_split)
    return model.score(X_val, y_val_split)

# Run Optuna Tuning
def tune_model(objective_fn, n_trials=30):
    study = optuna.create_study(direction='maximize')
    study.optimize(objective_fn, n_trials=n_trials)
    return study.best_params

params_lgb = tune_model(objective_lgb)
params_xgb = tune_model(objective_xgb)
params_svc = tune_model(objective_svc)

# ✅ Use clone() to ensure sklearn-compliant models for stacking
base_models = [
    ('xgb', clone(XGBSklearn(**params_xgb))),
    ('lgb', clone(LGBMSklearn(**params_lgb))),
    ('svc', clone(SVC(**params_svc)))
]

# Meta-Model Stacking
stacked_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=StratifiedKFold(n_splits=5),
    n_jobs=-1
)

# Train and Evaluate Final Model
stacked_model.fit(X_train, y_train_split)
y_val_pred = stacked_model.predict(valid_features)

print("\n📊 Final Validation Report:")
print(classification_report(y_valid, y_val_pred))
print("✅ Validation Accuracy:", accuracy_score(y_valid, y_val_pred))

# Save Final Model
joblib.dump(stacked_model, "optuna_stacked_meta_model.pkl")

In [ ]:
import os
import numpy as np
import tensorflow as tf
import gc
import optuna
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

# Paths
train_dir = r"data/train"
valid_dir = r"data/valid"

# Config
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 8
EPOCHS = 25

# Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    zoom_range=0.3,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)
valid_generator = valid_datagen.flow_from_directory(
    valid_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)

num_classes = len(train_generator.class_indices)
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
train_labels = train_generator.classes

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = dict(enumerate(class_weights))

early_stopping_cb = EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)

# Fine-tuned EfficientNetB3 model
def build_fine_tuned_model():
    base_model = EfficientNetB3(include_top=False, weights='imagenet', input_shape=input_shape)
    for layer in base_model.layers[:200]:
        layer.trainable = False
    for layer in base_model.layers[200:]:
        layer.trainable = True

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.6)(x)
    out = Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = Model(inputs=base_model.input, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    model.fit(train_generator, validation_data=valid_generator, epochs=EPOCHS,
              callbacks=[early_stopping_cb, reduce_lr], class_weight=class_weights, verbose=1)

    return model

# Train model and extract features
model = build_fine_tuned_model()
feature_model = Model(inputs=model.input, outputs=model.layers[-3].output)
train_features = feature_model.predict(train_generator, verbose=1)
valid_features = feature_model.predict(valid_generator, verbose=1)

# Preprocessing
scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
valid_features = scaler.transform(valid_features)

# Optional: skip PCA to retain all features
# from sklearn.decomposition import PCA
# pca = PCA(n_components=100)
# train_features = pca.fit_transform(train_features)
# valid_features = pca.transform(valid_features)

# Apply SMOTE
X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(train_features, train_labels)
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

# Optuna objective
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[early_stopping(stopping_rounds=10), log_evaluation(0)])
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='macro')

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# Train final model
best_params = study.best_params
best_params.update({'random_state': 42})
final_model = LGBMClassifier(**best_params)
final_model.fit(X_resampled, y_resampled)

# Evaluate
train_preds = final_model.predict(train_features)
valid_preds = final_model.predict(valid_features)

print("\n🔍 Training Report:")
print(classification_report(train_labels, train_preds, target_names=list(train_generator.class_indices.keys())))
print(f"Training Accuracy: {accuracy_score(train_labels, train_preds) * 100:.2f}%")

print("\n📊 Validation Report:")
print(classification_report(valid_generator.classes, valid_preds, target_names=list(valid_generator.class_indices.keys())))
print(f"Validation Accuracy: {accuracy_score(valid_generator.classes, valid_preds) * 100:.2f}%")

In [ ]:
import os
import numpy as np
import tensorflow as tf
import gc
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import EfficientNetB3, DenseNet121, InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from xgboost import XGBClassifier

# Paths
train_dir = r"data/train"
valid_dir = r"data/valid"

IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 8

# Data Generators
train_datagen = ImageDataGenerator(rescale=1./255)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)
valid_generator = valid_datagen.flow_from_directory(
    valid_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)

num_classes = len(train_generator.class_indices)
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
train_labels = train_generator.classes
valid_labels = valid_generator.classes

# Feature Extraction
def extract_features(model_class, model_name, generator):
    print(f"🔍 Extracting features from {model_name}...")
    tf.keras.backend.clear_session()
    base_model = model_class(include_top=False, weights='imagenet', input_shape=input_shape)
    base_model.trainable = False
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.5)(x)
    feature_model = Model(inputs=base_model.input, outputs=x)
    features = feature_model.predict(generator, verbose=1)
    del feature_model
    gc.collect()
    return features

# Extract features from all three CNNs
features_effnet_train = extract_features(EfficientNetB3, "EfficientNetB3", train_generator)
features_densenet_train = extract_features(DenseNet121, "DenseNet121", train_generator)
features_inception_train = extract_features(InceptionV3, "InceptionV3", train_generator)

features_effnet_valid = extract_features(EfficientNetB3, "EfficientNetB3", valid_generator)
features_densenet_valid = extract_features(DenseNet121, "DenseNet121", valid_generator)
features_inception_valid = extract_features(InceptionV3, "InceptionV3", valid_generator)

# Concatenate features
X_train_full = np.concatenate([features_effnet_train, features_densenet_train, features_inception_train], axis=1)
X_valid_full = np.concatenate([features_effnet_valid, features_densenet_valid, features_inception_valid], axis=1)

# Scale
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_valid_full = scaler.transform(X_valid_full)

y_train = train_labels
y_valid = valid_labels

# Train/Test split for Optuna
X_train, X_val, y_train_split, y_val_split = train_test_split(X_train_full, y_train, test_size=0.2, stratify=y_train, random_state=42)

# Optuna objective
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 42,
        'tree_method': 'gpu_hist'
    }
    model = XGBClassifier(**params, use_label_encoder=False, eval_metric='mlogloss')
    model.fit(X_train, y_train_split)
    preds = model.predict(X_val)
    return f1_score(y_val_split, preds, average='macro')

# Run Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# Final model
best_params = study.best_params
best_params.update({'random_state': 42, 'use_label_encoder': False, 'eval_metric': 'mlogloss', 'tree_method': 'gpu_hist'})
final_model = XGBClassifier(**best_params)
final_model.fit(X_train_full, y_train)

# Evaluate
train_preds = final_model.predict(X_train_full)
valid_preds = final_model.predict(X_valid_full)

print("\n📈 Training Report:")
print(classification_report(y_train, train_preds, target_names=list(train_generator.class_indices.keys())))
print(f"Training Accuracy: {accuracy_score(y_train, train_preds) * 100:.2f}%")

print("\n📊 Validation Report:")
print(classification_report(y_valid, valid_preds, target_names=list(valid_generator.class_indices.keys())))
print(f"Validation Accuracy: {accuracy_score(y_valid, valid_preds) * 100:.2f}%")

In [ ]:
!pip install tensorflow-addons

In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedShuffleSplit
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import optuna

# ---- CONFIG ----
train_dir = r"data/train"
valid_dir = r"data/valid"
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 8
EPOCHS = 25

# ---- DATA AUGMENTATION ----
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    zoom_range=0.3,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)

num_classes = len(train_generator.class_indices)
input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
train_labels = train_generator.classes

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = dict(enumerate(class_weights))

early_stopping_cb = EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)

# ---- CUSTOM FOCAL LOSS ----
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        epsilon = K.epsilon()
        y_pred = K.clip(y_pred, epsilon, 1. - epsilon)
        cross_entropy = -y_true * K.log(y_pred)
        weight = alpha * K.pow(1 - y_pred, gamma)
        loss = weight * cross_entropy
        return K.sum(loss, axis=1)
    return loss

# ---- BUILD MODEL ----
def build_fine_tuned_model():
    base_model = EfficientNetB3(include_top=False, weights='imagenet', input_shape=input_shape)
    for layer in base_model.layers[:200]:
        layer.trainable = False
    for layer in base_model.layers[200:]:
        layer.trainable = True

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.6)(x)
    out = Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = Model(inputs=base_model.input, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss=focal_loss(gamma=2.0, alpha=0.25),
                  metrics=['accuracy'])

    model.fit(train_generator, validation_data=valid_generator, epochs=EPOCHS,
              callbacks=[early_stopping_cb, reduce_lr], class_weight=class_weights, verbose=1)

    return model

# ---- TRAIN MODEL & EXTRACT FEATURES ----
model = build_fine_tuned_model()
feature_model = Model(inputs=model.input, outputs=model.layers[-3].output)

train_features = feature_model.predict(train_generator, verbose=1)
valid_features = feature_model.predict(valid_generator, verbose=1)

scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
valid_features = scaler.transform(valid_features)

# ---- SMOTE ----
X_resampled, y_resampled = SMOTE(random_state=42).fit_resample(train_features, train_labels)

# ---- STRATIFIED SPLIT ----
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, val_index in sss.split(X_resampled, y_resampled):
    X_train, X_val = X_resampled[train_index], X_resampled[val_index]
    y_train, y_val = y_resampled[train_index], y_resampled[val_index]

# ---- OPTUNA OBJECTIVE ----
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[early_stopping(stopping_rounds=10), log_evaluation(0)])
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='macro')

# ---- RUN OPTUNA ----
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# ---- FINAL LGBM MODEL ----
best_params = study.best_params
best_params.update({'random_state': 42})
final_model = LGBMClassifier(**best_params)
final_model.fit(X_resampled, y_resampled)

# ---- EVALUATION ----
train_preds = final_model.predict(train_features)
valid_preds = final_model.predict(valid_features)

print("\n🔍 Training Report:")
print(classification_report(train_labels, train_preds, target_names=list(train_generator.class_indices.keys())))
print(f"Training Accuracy: {accuracy_score(train_labels, train_preds) * 100:.2f}%")

print("\n📊 Validation Report:")
print(classification_report(valid_generator.classes, valid_preds, target_names=list(valid_generator.class_indices.keys())))
print(f"Validation Accuracy: {accuracy_score(valid_generator.classes, valid_preds) * 100:.2f}%")

In [ ]:
import tensorflow as tf
print(tf.__version__)